# P2 CHAIN A — club attendance (validate → sqlite → queries → Flask)

Closed-book mega-question. Reference voice: `DB.ipynb`, `WEB DEV.ipynb`. **Do not open them.**

**Rules**
- One kernel; run parts in order.
- Do **not** scroll to **Answer keys** until that part's checkpoint prints `PASS` (or you are marking).
- Later parts need earlier artifacts (`clean_members`, `clean_attendance`, `chain_a.db`).
- Templates live in `templates/chain_a/` — form `name=` and `url_for(...)` targets are **not** the usual latecoming ones.

**Data:** `MEMBERS.csv`, `ATTENDANCE.csv` in this folder (deliberately dirty rows included).

Suggested session: A1+A2 one day; A3 next; A4 when sqlite is cold.


---
## Part A1 — load + validate (12 min)

Read both CSVs. Skip header rows. Build:

- `clean_members`: list of `(member_id:int, name:str, house:str, year:int)`
- `clean_attendance`: list of `(date:str, member_id:int, session:str)`
- `rejected`: list of short reason strings for every rejected row

**Member rules:** `member_id` must parse as `int`; `name` non-empty; `house` non-empty; `year` in `{5, 6}`.

**Attendance rules:** `date` non-empty; `member_id` parses as `int`; `session` non-empty.

Do **not** require FK membership yet. Use explicit `if` checks — no bare `try/except` as your only validation.


In [1]:
# Part A1 — YOUR ATTEMPT
import csv

clean_members = []
clean_attendance = []
rejected = []

with open("ATTENDANCE.csv") as f:
    file = csv.reader(f)
    next(file)
    for row in file:
        if row[0] == '':
            rejected.append('empty date')
        elif row[1].isnumeric() == False:
            rejected.append('member_id not int')
        elif row[2] == '':
            rejected.append('session empty')
        else:
            row[1] = int(row[1])
            clean_attendance.append(row)
            
    
with open("MEMBERS.csv") as f:
    file = csv.reader(f)
    next(file)
    for row in file:
        if row[0].isnumeric() == False:
            rejected.append('member_id not int')
        elif row[1] == '':
            rejected.append('name empty')
        elif row[2] == '':
            rejected.append('house empty')
        elif int(row[3]) not in (5,6):
            rejected.append('invalid years')
        else:
            row[0] = int(row[0])
            row[3] = int(row[3])
            clean_members.append(row)


In [2]:
# CHECKPOINT A1 — run after your attempt (not the answer key)
ok = True
msgs = []
if len(clean_members) != 15:
    ok = False
    msgs.append(f"clean_members expected 15, got {len(clean_members)}")
if len(clean_attendance) != 33:
    ok = False
    msgs.append(f"clean_attendance expected 33, got {len(clean_attendance)}")
if len(rejected) < 3:
    ok = False
    msgs.append(f"rejected expected at least 3, got {len(rejected)}")
ids = {m[0] for m in clean_members}
if 1013 in ids or 1014 in ids:
    ok = False
    msgs.append("bad member ids leaked into clean_members")
print("PASS" if ok else "FAIL: " + "; ".join(msgs))


PASS


---
## Part A2 — schema + seed (12 min)

Connect to **`chain_a.db`**. Set `row_factory = sqlite3.Row`. DROP then CREATE:

- `Member(member_id INTEGER PRIMARY KEY, name TEXT, house TEXT, year INTEGER)`
- `Attendance(date TEXT, member_id INTEGER, session TEXT, PRIMARY KEY(date, member_id), FOREIGN KEY(member_id) REFERENCES Member(member_id))`

Seed from `clean_members` / `clean_attendance` with `?` placeholders and `INSERT OR IGNORE`. `commit()`.

**Check target:** 15 members; attendance rows whose `member_id` exists in Member.


In [3]:
# Part A2 — YOUR ATTEMPT
import sqlite3

conn = sqlite3.connect("chain_a.db")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

#reset db
cur.execute("""
DROP TABLE IF EXISTS Member
""")
cur.execute("""
DROP TABLE IF EXISTS Attendance 
""")

#design schema
cur.execute("""
CREATE TABLE Member (
member_id INTEGER PRIMARY KEY,
name TEXT,
house TEXT,
year INTEGER
)
""")
cur.execute("""
CREATE TABLE Attendance (
date TEXT,
member_id INTEGER,
session TEXT,
PRIMARY KEY(date, member_id)
FOREIGN KEY(member_id) REFERENCES Member(member_id)
)
""")

#seed data
for row in clean_members:
    cur.execute("""
    INSERT OR IGNORE INTO Member (member_id, name, house, year) 
    VALUES (?,?,?,?)
    """, (row[0], row[1], row[2], row[3]))
for row in clean_attendance:
    cur.execute("""
    INSERT OR IGNORE INTO Attendance (date, member_id, session) 
    VALUES (?,?,?)
    """, (row[0], row[1], row[2]))
conn.commit()
conn.close()

In [4]:
# CHECKPOINT A2
import sqlite3
conn = sqlite3.connect("chain_a.db")
conn.row_factory = sqlite3.Row
cur = conn.cursor()
ok = True
msgs = []
try:
    n_m = cur.execute("SELECT COUNT(*) AS c FROM Member").fetchone()["c"]
    n_a = cur.execute("SELECT COUNT(*) AS c FROM Attendance").fetchone()["c"]
    if n_m != 15:
        ok = False
        msgs.append(f"Member count {n_m} != 15")
    if n_a < 30:
        ok = False
        msgs.append(f"Attendance count {n_a} looks too low")
    row = cur.execute("SELECT name FROM Member WHERE member_id = ?", (1001,)).fetchone()
    if row is None or row["name"] != "Ada Tan":
        ok = False
        msgs.append("row_factory / seed smoke failed for 1001")
except Exception as e:
    ok = False
    msgs.append(str(e))
conn.close()
print("PASS" if ok else "FAIL: " + "; ".join(msgs))


PASS


---
## Part A3 — query bank (15 min)

Using `chain_a.db`, write parameterised queries (wildcards in the **tuple**):

1. **JOIN hit list:** members in house `East` who attended a `Training` session — print `name`, `date`, `session`.
2. **HAVING:** houses with `COUNT(*)` of attendance rows `>= 5` — print `house`, total.
3. **Never-attended:** members with **no** Attendance row (`LEFT OUTER JOIN` + `IS NULL`) in house `West` — print names.
4. One **UPDATE** then **DELETE** on a throwaway row you `INSERT` first (parameterised); `commit`.

Remember: aggregates belong in `HAVING`, not `WHERE`.


In [5]:
# Part A3 — YOUR ATTEMPT
import sqlite3
conn = sqlite3.connect("chain_a.db")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

# 1 JOIN

cur.execute("""
SELECT Member.name, Attendance.date, Attendance.session 
FROM Member INNER JOIN Attendance ON Member.member_id = Attendance.member_id
WHERE Member.house = ?
ORDER BY Member.name ASC
""", ('East',))
for row in cur.fetchall():
    print(row['name'], row['date'], row['session'])

# 2 HAVING

cur.execute("""
SELECT Member.house, COUNT(*) as total
FROM Member INNER JOIN Attendance ON Member.member_id = Attendance.member_id
GROUP BY Member.house
HAVING COUNT(*) >= ?
ORDER BY total DESC
""", (5,))
for row in cur.fetchall():
    print(row['house'], row['total'])
    
# 3 never-attended

cur.execute("""
SELECT Member.name 
FROM Member LEFT OUTER JOIN Attendance ON Member.member_id = Attendance.member_id
WHERE Member.house = ? AND Attendance.member_id IS NULL
""", ('West',))
for row in cur.fetchall():
    print(row['name'])

# 4 CRUD smoke

cur.execute("""
INSERT INTO Member (member_id, name, house, year)
VALUES (?,?,?,?)
""", (1030, 'nicer person', 'West', 6))
conn.commit()
cur.execute("""
UPDATE Member 
SET name = ?, year = ?
WHERE member_id = ?
""", ('ninja', 5,1070))
conn.commit()
cur.execute("""
DELETE FROM Member
WHERE member_id = ?
""", (1070,))
conn.commit()
conn.close()


Ada Tan 2025-01-10 Training
Ada Tan 2025-01-11 Training
Ada Tan 2025-01-13 Training
Ada Tan 2025-01-17 Match
Ada Tan 2025-01-22 Training
Cara Lim 2025-01-10 Match
Cara Lim 2025-01-14 Training
Cara Lim 2025-01-26 Training
Gina Ho 2025-01-13 Match
Gina Ho 2025-01-20 Training
Kai Wong 2025-01-17 Training
Kai Wong 2025-01-28 Match
Omar Ali 2025-01-19 Training
Omar Ali 2025-01-30 Training
East 14
North 8
West 6
South 5


In [6]:
# CHECKPOINT A3
import sqlite3
conn = sqlite3.connect("chain_a.db")
conn.row_factory = sqlite3.Row
cur = conn.cursor()
ok = True
msgs = []
east_train = cur.execute(
    "SELECT Member.name, Attendance.date, Attendance.session "
    "FROM Member INNER JOIN Attendance ON Member.member_id = Attendance.member_id "
    "WHERE Member.house = ? AND Attendance.session = ?",
    ("East", "Training"),
).fetchall()
if len(east_train) < 1:
    ok = False
    msgs.append("JOIN East/Training returned nothing — DB empty or wrong schema?")
hav = cur.execute(
    "SELECT Member.house, COUNT(*) AS total "
    "FROM Member INNER JOIN Attendance ON Member.member_id = Attendance.member_id "
    "GROUP BY Member.house HAVING COUNT(*) >= 5"
).fetchall()
if len(hav) < 1:
    ok = False
    msgs.append("HAVING query returned no groups")
never = cur.execute(
    "SELECT Member.name FROM Member LEFT OUTER JOIN Attendance "
    "ON Member.member_id = Attendance.member_id "
    "WHERE Attendance.member_id IS NULL AND Member.house = ?",
    ("West",),
).fetchall()
_ = never
conn.close()
print("PASS" if ok else "FAIL: " + "; ".join(msgs))


PASS


---
## Part A4 — Flask wiring (20 min)

Build a Flask app that uses **`chain_a.db`** and templates under `templates/chain_a/`:

| Route | Function name | Template | Behaviour |
|-------|---------------|----------|-----------|
| `/` GET+POST | `portal` | `chain_a/portal.html` | POST reads `session_type` + `member_name`; validate with explicit `if`; **return** `redirect(url_for("hits", ...))` |
| `/hits/<session_type>/<member_name>` | `hits` | `chain_a/hits.html` | JOIN query; `LIKE` on name with wildcards in tuple; pass `rows`, `session_type`, `member_name` |
| `/log` GET+POST | `log_absence` | `chain_a/log.html` | form keys `mid`, `when`, `sess`; INSERT parameterised; redirect to `portal` |

Also: `get_db()`, `row_factory`, `if __name__ == "__main__": app.run(port=5001)` (5001 avoids clashing with WEB DEV on 5000).

Stop other Flask apps first. Checkpoint is structural only — also self-check by starting the app.


In [10]:
# Part A4 — YOUR ATTEMPT
from flask import Flask, render_template, request, redirect, url_for
import sqlite3

app = Flask(__name__)

# get_db, portal, hits, log_absence, __main__

def get_db():
    conn = sqlite3.connect("chain_a.db")
    conn.row_factory = sqlite3.Row
    return conn

accept = ['GET', 'POST']

@app.route("/", methods = accept)
def portal():
    error = None
    if request.method == 'POST':
        session_type = request.form.get("sess",'').strip()
        member_name = request.form.get("member_name",'').strip()
        if Training and Match and member_name:
            return redirect(url_for('hits', session_type = session_type, member_name = member_name))
        else:
            error = "Please fill in all fields"
    return render_template('chain_a/portal.html', error = error)

@app.route("/hits/<session_type>/<member_name>", methods = accept)
def hits(session_type, member_name):
    conn = get_db()
    cur = conn.cursor()
    cur.execute("""
    SELECT Member.name, Attendance.date, Attendance.session, Member.house 
    FROM Member INNER JOIN Attendance ON Member.member_id = Attendance.member_id
    WHERE Member.name = ? AND Attendance.session = ?
    ORDER BY Member.name ASC
    """,(f"%{member_name}%",session_type))
    conn.close()
    return render_template('chain_a/hits.html', rows = cur.fetchall(), session_type = session_type, member_name = member_name)
    
@app.route("/log", methods = accept)
def log_absence():
    error = None
    if request.method == 'POST':
        session_type = form.request.get("sess",'')
        member_id = form.request.get("mid",'')
        date = form.request.get("when",'')
        if session_type and member_name and date:
            conn = get_db()
            cur = conn.cursor()
            cur.execute("""
            INSERT OR IGNORE INTO Attendance
            VALUES(?,?,?)
            """, (date, member_id, session_type))
            conn.commit()
            conn.close()
            return redirect(url_for('portal'))
        else:
            error = "Please fill in all fields"
    else:
        return render_template('chain_a/log.html', error = error)

if __name__ == "__main__":
    app.run(port=5001)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit


In [12]:
# CHECKPOINT A4 — structural (does not start the server)
ok = True
msgs = []
needed = ["app", "get_db", "portal", "hits", "log_absence"]
g = globals()
for name in needed:
    if name not in g:
        ok = False
        msgs.append(f"missing {name}")
if "app" in g:
    rules = {rule.rule: rule.endpoint for rule in app.url_map.iter_rules() if rule.endpoint != "static"}
    expect = {"/": "portal", "/log": "log_absence"}
    for path, ep in expect.items():
        if rules.get(path) != ep:
            ok = False
            msgs.append(f"route {path} expected endpoint {ep}, got {rules.get(path)}")
    if "hits" not in rules.values():
        ok = False
        msgs.append("hits endpoint not registered")
print("PASS" if ok else "FAIL: " + "; ".join(msgs))
print("Manual: run app on 5001 and POST a search + log one absence.")


PASS
Manual: run app on 5001 and POST a search + log one absence.


---
# Answer keys — scroll only after checkpoints (or when marking)

## Answer — Part A1


In [ ]:
# Answer — Part A1
import csv

clean_members = []
clean_attendance = []
rejected = []

with open("MEMBERS.csv", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        if len(row) < 4:
            rejected.append("member: short row")
            continue
        mid_s, name, house, year_s = row[0].strip(), row[1].strip(), row[2].strip(), row[3].strip()
        if not mid_s.isdigit():
            rejected.append(f"member: bad id {mid_s!r}")
            continue
        if name == "" or house == "":
            rejected.append(f"member: empty name/house id={mid_s}")
            continue
        if not year_s.isdigit() or int(year_s) not in (5, 6):
            rejected.append(f"member: bad year {year_s!r}")
            continue
        clean_members.append((int(mid_s), name, house, int(year_s)))

with open("ATTENDANCE.csv", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        if len(row) < 3:
            rejected.append("attendance: short row")
            continue
        date, mid_s, session = row[0].strip(), row[1].strip(), row[2].strip()
        if date == "":
            rejected.append("attendance: empty date")
            continue
        if not mid_s.isdigit():
            rejected.append(f"attendance: bad id {mid_s!r}")
            continue
        if session == "":
            rejected.append("attendance: empty session")
            continue
        clean_attendance.append((date, int(mid_s), session))

print(len(clean_members), len(clean_attendance), len(rejected))
# traps: next(reader) for header; year domain {5,6}; empty checks before int()


## Answer — Part A2


In [ ]:
# Answer — Part A2
import sqlite3

conn = sqlite3.connect("chain_a.db")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS Attendance")
cur.execute("DROP TABLE IF EXISTS Member")
cur.execute(
    "CREATE TABLE Member("
    "member_id INTEGER PRIMARY KEY, name TEXT, house TEXT, year INTEGER)"
)
cur.execute(
    "CREATE TABLE Attendance("
    "date TEXT, member_id INTEGER, session TEXT, "
    "PRIMARY KEY(date, member_id), "
    "FOREIGN KEY(member_id) REFERENCES Member(member_id))"
)

for mid, name, house, year in clean_members:
    cur.execute(
        "INSERT OR IGNORE INTO Member(member_id, name, house, year) VALUES (?,?,?,?)",
        (mid, name, house, year),
    )
for date, mid, session in clean_attendance:
    cur.execute(
        "INSERT OR IGNORE INTO Attendance(date, member_id, session) VALUES (?,?,?)",
        (date, mid, session),
    )
conn.commit()
print(cur.execute("SELECT COUNT(*) AS c FROM Member").fetchone()["c"])
print(cur.execute("SELECT COUNT(*) AS c FROM Attendance").fetchone()["c"])
conn.close()
# traps: DROP child before parent; ? placeholders; commit


## Answer — Part A3


In [ ]:
# Answer — Part A3
import sqlite3
conn = sqlite3.connect("chain_a.db")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute(
    "SELECT Member.name, Attendance.date, Attendance.session "
    "FROM Member INNER JOIN Attendance ON Member.member_id = Attendance.member_id "
    "WHERE Member.house = ? AND Attendance.session = ? ORDER BY Attendance.date",
    ("East", "Training"),
)
for row in cur.fetchall():
    print(row["name"], row["date"], row["session"])

cur.execute(
    "SELECT Member.house, COUNT(*) AS total "
    "FROM Member INNER JOIN Attendance ON Member.member_id = Attendance.member_id "
    "GROUP BY Member.house HAVING COUNT(*) >= 5"
)
for row in cur.fetchall():
    print(row["house"], row["total"])

cur.execute(
    "SELECT Member.name FROM Member LEFT OUTER JOIN Attendance "
    "ON Member.member_id = Attendance.member_id "
    "WHERE Attendance.member_id IS NULL AND Member.house = ?",
    ("West",),
)
for row in cur.fetchall():
    print(row["name"])

cur.execute(
    "INSERT OR IGNORE INTO Attendance(date, member_id, session) VALUES (?,?,?)",
    ("2099-01-01", 1001, "Training"),
)
cur.execute(
    "UPDATE Attendance SET session = ? WHERE date = ? AND member_id = ?",
    ("Match", "2099-01-01", 1001),
)
cur.execute(
    "DELETE FROM Attendance WHERE date = ? AND member_id = ?",
    ("2099-01-01", 1001),
)
conn.commit()
conn.close()
# traps: HAVING not WHERE for COUNT; trailing comma on 1-tuples


## Answer — Part A4


In [ ]:
# Answer — Part A4
from flask import Flask, render_template, request, redirect, url_for
import sqlite3

app = Flask(__name__)

def get_db():
    conn = sqlite3.connect("chain_a.db")
    conn.row_factory = sqlite3.Row
    return conn

@app.route("/", methods=["GET", "POST"])
def portal():
    error = None
    if request.method == "POST":
        session_type = request.form.get("session_type", "").strip()
        member_name = request.form.get("member_name", "").strip()
        if session_type and member_name:
            return redirect(url_for("hits", session_type=session_type, member_name=member_name))
        error = "Please fill in all fields"
    return render_template("chain_a/portal.html", error=error)

@app.route("/hits/<session_type>/<member_name>")
def hits(session_type, member_name):
    conn = get_db()
    cur = conn.cursor()
    cur.execute(
        "SELECT Member.name, Member.house, Attendance.date, Attendance.session "
        "FROM Member INNER JOIN Attendance ON Member.member_id = Attendance.member_id "
        "WHERE Attendance.session = ? AND Member.name LIKE ? ORDER BY Attendance.date",
        (session_type, f"%{member_name}%"),
    )
    rows = cur.fetchall()
    conn.close()
    return render_template(
        "chain_a/hits.html",
        rows=rows,
        session_type=session_type,
        member_name=member_name,
    )

@app.route("/log", methods=["GET", "POST"])
def log_absence():
    error = None
    if request.method == "POST":
        mid = request.form.get("mid", "").strip()
        when = request.form.get("when", "").strip()
        sess = request.form.get("sess", "").strip()
        if not (mid and when and sess):
            error = "Please fill in all fields"
        else:
            conn = get_db()
            cur = conn.cursor()
            cur.execute(
                "INSERT OR IGNORE INTO Attendance(date, member_id, session) VALUES (?,?,?)",
                (when, mid, sess),
            )
            conn.commit()
            conn.close()
            return redirect(url_for("portal"))
    return render_template("chain_a/log.html", error=error)

if __name__ == "__main__":
    app.run(port=5001)
# traps: MUST return redirect; form keys match template; url_for endpoint == def name
